In [ ]:
import os
import glob
import numpy as np
import open3d as o3d
from plyfile import PlyData, PlyElement

# ==========================================
# CONFIGURATION
# ==========================================
# REPLACE these paths with your actual directories
input_dir = r"F:\Segmentation_training\RC_bridges\training_datasets\correct_instance_labeled datasets_Copy"
output_dir = r"F:\Segmentation_training\RC_bridges\training_datasets\correct_instance_labeled datasets_with_normals"

# Create output directory to ensure we don't overwrite original files
os.makedirs(output_dir, exist_ok=True)

# Find all PLY files
ply_files = glob.glob(os.path.join(input_dir, "*.ply"))
if not ply_files:
    raise FileNotFoundError(f"No .ply files found in {input_dir}")

print(f"Found {len(ply_files)} PLY files. Beginning processing...\n")

# ==========================================
# PHASE 1: AUTOMATIC RADIUS ESTIMATION
# ==========================================
print("--- Phase 1: Estimating Optimal Search Radius ---")
first_file = ply_files[0]
temp_plydata = PlyData.read(first_file)
temp_vertex_data = temp_plydata['vertex'].data

# Extract X, Y, Z to calculate nearest neighbor distance
sample_xyz = np.vstack([temp_vertex_data['x'], temp_vertex_data['y'], temp_vertex_data['z']]).T

# Use Open3D to compute average nearest neighbor distance
pcd_sample = o3d.geometry.PointCloud()
pcd_sample.points = o3d.utility.Vector3dVector(sample_xyz)
distances = pcd_sample.compute_nearest_neighbor_distance()
d_avg = np.mean(distances)

# ==========================================
# RECALIBRATED PARAMETERS FOR BRIDGE GEOMETRY
# ==========================================
# Hard physical radius cap (30 cm) to prevent cross-component bleeding
optimal_radius = 0.30  # meters

# 12 to 16 points is optimal for plane fitting without corner-blurring
optimal_max_nn = 16

print(f"Using Physical Search Radius: {optimal_radius}m")
print(f"Using Maximum Neighbor Cap (max_nn): {optimal_max_nn} points\n")


# ==========================================
# PHASE 2: BATCH PROCESSING
# ==========================================
print("--- Phase 2: Processing Point Clouds ---")

for idx, file_path in enumerate(ply_files, 1):
    file_name = os.path.basename(file_path)
    save_path = os.path.join(output_dir, file_name)
    
    # 1. Read strict PLY structure
    plydata = PlyData.read(file_path)
    old_data = plydata['vertex'].data
    old_names = old_data.dtype.names
    
    # Extract XYZ
    xyz = np.vstack([old_data['x'], old_data['y'], old_data['z']]).T
    
    # 2. Compute Normals using Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius = optimal_radius, max_nn = optimal_max_nn))
    
    # Ensure normals point upwards consistently towards the camera (sky)
    pcd.orient_normals_towards_camera_location(camera_location=[0, 0, 1000])
    normals = np.asarray(pcd.normals)
    
    # 3. Compute Relative Z (z_rel)
    z_rel = xyz[:, 2] - np.min(xyz[:, 2])
    
    # 4. Build the new structured data type retaining all original columns
    new_dtype_list = []
    
    # Copy old datatypes (skip nx, ny, nz, z_rel if they already exist so we can cleanly overwrite)
    for name in old_names:
        if name not in ['nx', 'ny', 'nz', 'z_rel']:
            new_dtype_list.append((name, old_data.dtype[name]))
            
    # Append our new geometric feature datatypes as 32-bit floats
    new_dtype_list.extend([
        ('nx', 'f4'),
        ('ny', 'f4'),
        ('nz', 'f4'),
        ('z_rel', 'f4')
    ])
    
    # Create empty structured numpy array with the new datatypes
    new_data = np.empty(len(old_data), dtype=np.dtype(new_dtype_list))
    
    # 5. Populate the new array
    # Map original data
    for name in old_names:
        if name not in ['nx', 'ny', 'nz', 'z_rel']:
            new_data[name] = old_data[name]
            
    # Map newly computed geometric data
    new_data['nx'] = normals[:, 0]
    new_data['ny'] = normals[:, 1]
    new_data['nz'] = normals[:, 2]
    new_data['z_rel'] = z_rel
    
    # 6. Write to the new PLY file in binary tensor format
    el = PlyElement.describe(new_data, 'vertex')
    PlyData([el], text=False).write(save_path)
    
    # Print progress
    if idx % 10 == 0 or idx == len(ply_files):
        print(f"Processed [{idx}/{len(ply_files)}]: {file_name}")

print("\n==========================================")
print(f"SUCCESS! All {len(ply_files)} files processed.")
print(f"Features added: 'nx', 'ny', 'nz', 'z_rel'")
print(f"Saved securely in: {output_dir}")
print("==========================================")

Found 133 PLY files. Beginning processing...

--- Phase 1: Estimating Optimal Search Radius ---
Using Physical Search Radius: 0.3m
Using Maximum Neighbor Cap (max_nn): 16 points

--- Phase 2: Processing Point Clouds ---
Processed [10/133]: Bridge CG 02_instance_cloud_crop_2.ply
Processed [20/133]: Bridge CG 05_instance_cloud_crop_2.ply
Processed [30/133]: Bridge CG 07_instance_cloud_crop_2.ply
Processed [40/133]: Bridge GB 01_instance_cloud_crop_2.ply
Processed [50/133]: Bridge GB 02_instance_cloud_crop_8.ply
Processed [60/133]: Bridge GB 03_instance_cloud_crop_8.ply
Processed [70/133]: Bridge GB 05_instance_cloud_crop_1.ply
Processed [80/133]: Bridge GB 08_instance_cloud_crop_0.ply
Processed [90/133]: Bridge GB 11_instance_cloud.ply
Processed [100/133]: Bridge SB 01_instance_cloud_crop_2.ply
Processed [110/133]: Bridge SB 03_instance_cloud.ply
Processed [120/133]: Bridge SB 04_instance_cloud_crop_3.ply
Processed [130/133]: Culvert_Arch_instance_cloud_crop_0.ply
Processed [133/133]: Cu

In [ ]:
'''
import os
import glob
import open3d as o3d
import numpy as np

# 1. Define input and output folders
input_dir = r"F:\Segmentation_training\RC_bridges_raw"
output_dir = r"F:\Segmentation_training\RC_bridges_processed"
os.makedirs(output_dir, exist_ok=True)

ply_files = glob.glob(os.path.join(input_dir, "*.ply"))
print(f"Found {len(ply_files)} PLY files to process...")

for file_path in ply_files:
    file_name = os.path.basename(file_path)
    save_path = os.path.join(output_dir, file_name)
    
    # Read PLY file
    pcd = o3d.io.read_point_cloud(file_path)
    
    # Estimate surface normals
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.2, max_nn=30)
    )
    # Orient normals consistently towards upper Z
    pcd.orient_normals_towards_camera_location(camera_location=[0, 0, 1000])
    
    # Save processed point cloud with normals into the new directory
    o3d.io.write_point_cloud(save_path, pcd)

print(f"Processing complete! Enhanced files saved to: {output_dir}")

In [2]:
## IMPROVED CODE


import os
import glob
import numpy as np
import open3d as o3d
from plyfile import PlyData, PlyElement

# ==========================================
# CONFIGURATION
# ==========================================
input_dir = r"F:\Segmentation_training\RC_bridges\training_datasets\correct_instance_labeled datasets_only_culverts_and_cross-girders"
output_dir = r"F:\Segmentation_training\RC_bridges\training_datasets\correct_instance_labeled datasets_only_culverts_and_cross-girders_with_surf_normals"

os.makedirs(output_dir, exist_ok=True)

ply_files = glob.glob(os.path.join(input_dir, "*.ply"))
if not ply_files:
    raise FileNotFoundError(f"No .ply files found in {input_dir}")

print(f"Found {len(ply_files)} PLY files. Beginning processing...\n")

# Optimal KNN for 40k point bridge clouds
K_NEIGHBORS = 30  # Standard for clean plane fitting without smoothing sharp edges

for idx, file_path in enumerate(ply_files, 1):
    file_name = os.path.basename(file_path)
    save_path = os.path.join(output_dir, file_name)
    
    # 1. Read PLY file
    plydata = PlyData.read(file_path)
    old_data = plydata['vertex'].data
    old_names = old_data.dtype.names
    
    # Extract XYZ
    xyz = np.vstack([old_data['x'], old_data['y'], old_data['z']]).T
    
    # 2. Compute Surface Normals using Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    
    # KNN search ensures dense & sparse regions both get 30 points
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamKNN(knn=K_NEIGHBORS)
    )
    
    # Orient normals consistently along local surface geometry (preserves top vs bottom faces)
    pcd.orient_normals_consistent_tangent_plane(k=K_NEIGHBORS)
    normals = np.asarray(pcd.normals)
    
    # Safety: Clean NaNs and enforce unit length (norm = 1.0)
    norms = np.linalg.norm(normals, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    normals = normals / norms
    normals = np.nan_to_num(normals, nan=0.0)
    
    # 3. Compute Per-Cloud Min-Max Normalized z_rel in [0, 1]
    z_vals = xyz[:, 2]
    z_min, z_max = np.min(z_vals), np.max(z_vals)
    z_range = z_max - z_min
    z_rel_norm = (z_vals - z_min) / (z_range if z_range > 1e-6 else 1.0)
    
    # 4. Construct new structured array schema
    new_dtype_list = []
    for name in old_names:
        if name not in ['nx', 'ny', 'nz', 'z_rel']:
            new_dtype_list.append((name, old_data.dtype[name]))
            
    new_dtype_list.extend([
        ('nx', 'f4'),
        ('ny', 'f4'),
        ('nz', 'f4'),
        ('z_rel', 'f4')
    ])
    
    new_data = np.empty(len(old_data), dtype=np.dtype(new_dtype_list))
    
    # 5. Populate structured array
    for name in old_names:
        if name not in ['nx', 'ny', 'nz', 'z_rel']:
            new_data[name] = old_data[name]
            
    new_data['nx'] = normals[:, 0].astype(np.float32)
    new_data['ny'] = normals[:, 1].astype(np.float32)
    new_data['nz'] = normals[:, 2].astype(np.float32)
    new_data['z_rel'] = z_rel_norm.astype(np.float32)
    
    # 6. Save binary PLY
    el = PlyElement.describe(new_data, 'vertex')
    PlyData([el], text=False).write(save_path)
    
    if idx % 5 == 0 or idx == len(ply_files):
        print(f"Processed [{idx}/{len(ply_files)}]: {file_name}")

print("\n==========================================")
print(f"SUCCESS! All {len(ply_files)} PLY files generated with clean 7D feature sets.")
print(f"Saved in: {output_dir}")
print("==========================================")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Found 41 PLY files. Beginning processing...

Processed [5/41]: Bridge CG 01_instance_cloud_crop_3.ply
Processed [10/41]: Bridge CG 02_instance_cloud_crop_2.ply
Processed [15/41]: Bridge CG 04_instance_cloud_crop_2.ply
Processed [20/41]: Bridge CG 05_instance_cloud_crop_2.ply
Processed [25/41]: Bridge CG 05_instance_cloud_crop_7.ply
Processed [30/41]: Bridge CG 07_instance_cloud_crop_2.ply
Processed [35/41]: Bridge CG 07_instance_cloud_crop_7.ply
Processed [40/41]: Culvert_Box_instance_cloud_crop_0.ply
Processed [41/41]: Culvert_Box_instance_cloud_crop_1.ply

SUCCESS! All 41 PLY files generated with clean 7D feature sets.
Saved in: F:\Segmentation_training\RC_bridges\training_datasets\correct_instance_labeled datasets_only_culverts_and_cross-girders_with_surf_normals
